# Welcome to the INTSYCURE Jupyter Notebook

This will be the primary jupyter notebook for us to do our feature engineering and model training. Kindly follow each section when modifying this notebook to make things easier. Feel free to make subsections if needed.

## README

This code's main branch will be in `mco2`. For creating your own branch, kindly format the branch name with `mco2-<name>` (first name or last name up to you). Branch off from this `mco2` branch, make your changes, and the changes can be later pulled into the main branch or specific features/code can be manaully taken from each of our notebooks, since the jupyter notebook format may be a pain in the azz to work with in git.

## Import Modules

Kindly install these modules if required.

In [45]:
# Installing packages (run in cli):
#   pip install <package name 

import pandas as pd

## Importing the Data
The `master_dataset.csv` file is imported, ready to be used.

In [46]:
master_data = pd.read_csv('master_dataset.csv')
master_data.head

<bound method NDFrame.head of       sentence_id  word_id                                           sentence  \
0               0        0  Kaya kayong mga babae wag kayong basta basta m...   
1               0        1  Kaya kayong mga babae wag kayong basta basta m...   
2               0        2  Kaya kayong mga babae wag kayong basta basta m...   
3               0        3  Kaya kayong mga babae wag kayong basta basta m...   
4               0        4  Kaya kayong mga babae wag kayong basta basta m...   
...           ...      ...                                                ...   
9631          499     9631  Hello po mag ask po ako sa inyo ng help para p...   
9632          499     9632  Hello po mag ask po ako sa inyo ng help para p...   
9633          499     9633  Hello po mag ask po ako sa inyo ng help para p...   
9634          499     9634  Hello po mag ask po ako sa inyo ng help para p...   
9635          499     9635  Hello po mag ask po ako sa inyo ng help para p...  

## Feature Engineering

In this section, we will engineer each of the features for the model to be trained.

### Feature Tracking
Consider this as a checklist of the features that I am considering
- [x] has consecutive a/i/u  ; ENG words often only have consecutive Os or Es but not usually A, I, or U
- [x] has x  ; does not appear often in FIL words
- [x] has z  ; does not appear often in FIL words
- [x] has a repeating prefix  ; i.e. **kaka**in
- [x] has a repeating first char  ; i.e. **uupo**
- [x] has number  ; bc all numbers SHOULD be OTH (NOTE: we will probably have to clean the data for this)
- [x] has special char  ; bc all numbers SHOULD be OTH (NOTE: we will probably have to clean the data for this)

In [47]:
# creating a copy of the master dataset for feature

df = master_data.copy()

### Feature: Has Consecutive A/I/U

English words may have consecutive Os (boom) or Es (tree) but not usually A, I, or U. Filipino may have words such as **kaakbay** (consecutive As) **ginigiit** (consecutive Is) or **uupo** (consecutive Us)

In [48]:
df['consec_aui'] = False

for index, row in df.iterrows():
    word = row.loc['word'].strip()

    found = True
    prev_char = None

    char_checks = ['a', 'i', 'u']

    for char in word:
        if prev_char == char and char in char_checks:
            df.loc[index, 'consec_aui'] = True
            found = True
            break
        prev_char = char

### Has X, Has Z
Both X and Z are not common or are not present at all in Filipino words.

In [49]:
df['has_x'] = False
df['has_z'] = False

for index, row in df.iterrows():
    word = row.loc['word'].strip()

    for char in word:
        match char:
            case 'x':
                df.loc[index, 'has_x'] = True
            case 'z':
                df.loc[index, 'has_z'] = True

### Has Repeating prefix
Looks for words with a repeating start i.e. **kakakain** or **mamaya**

NOTE: does not cover words with a repeating first letter. That will be a separate feature to clarify their difference. This will also only check for prefixes with length 2-4.

In [50]:
df['rpt_prfx'] = False

for index, row in df.iterrows():
    word = row.loc['word'].strip()

    for i in range(2,5):
        if word[0:i] == word[i:i+i]:
            df.loc[index, 'rpt_prfx'] = True

### Has Repeating First Letter
Looks for words with a repeating first letter i.e. **uupo**

In [53]:
df['rpt_fchr'] = False

for index, row in df.iterrows():
    word = row.loc['word'].strip()

    if word[0:1] == word[1:2]:
        df.loc[index, 'rpt_prfx'] = True

### Has Number or Special Character
Looks for any number in the word. Another feature is for any special character being in the word.

In [58]:
df['has_num'] = False
df['has_spec'] = False

special_characters = [
    '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', 
    '+', ',', '-', '.', '/', ':', ';', '<', '=', '>', 
    '?', '@', '[', '\\', ']', '^', '_', '`', '{', '|', 
    '}', '~'
]

numbers = [str(i) for i in range(0, 10)]

for index, row in df.iterrows():
    word = row.loc['word'].strip()

    for char in word:
        if char in numbers:
            df.loc[index, 'has_num'] = True
        if char in special_characters:
            df.loc[index, 'has_spec'] = True

In [75]:
# Code to see avg vowel ratios

df2 = df.copy()
df2['vowel_ratio'] = 0.00
df2['consonant_ratio'] = 0.00

vowels = ['a', 'e', 'i', 'o', 'u']
consonants = "bcdfghjklmnpqrstvwxyz"


for index, row in df2.iterrows():
    word = row.loc['word'].strip()
    vowel_count = 0
    consonant_count = 0
    for char in word:
        if char in vowels:
            vowel_count += 1
        elif char in consonants:
            consonant_count += 1 
    df2.loc[index, 'vowel_ratio'] = vowel_count/len(word)
    df2.loc[index, 'consonant_ratio'] = consonant_count/len(word)


avg_vowels_df = df2.groupby('annot')['vowel_ratio'].mean().reset_index()
avg_consonants_df = df2.groupby('annot')['consonant_ratio'].mean().reset_index()

print(avg_vowels_df)
print(avg_consonants_df)

# Doesn't seem very significant, but it could be a bit skewed...

  annot  vowel_ratio
0    CS     0.400910
1   ENG     0.365529
2   FIL     0.399606
3   OTH     0.132623
  annot  consonant_ratio
0    CS         0.539142
1   ENG         0.577262
2   FIL         0.580539
3   OTH         0.160003


## Model Training
This section will contain the main code for training and exporting our model.

In [ ]:
# view the features 
df.columns

Index(['sentence_id', 'word_id', 'sentence', 'word', 'annot', 'consec_aui',
       'has_x', 'has_z', 'rpt_prfx', 'has_num', 'rpt_fchr', 'has_spec'],
      dtype='object')